# Restaurant Annual Turnover Prediction — October 18 model comparison

**Student register number:** submission

**Student name:** submission

**Team name:** submission

In Google Colab, upload this notebook, select a CPU runtime and run all cells.
Upload the four official input CSVs when prompted. The final cell downloads the
500-row prediction file. Save the executed notebook from Colab's File menu.
Repeated validation may take several minutes; progress is printed per split.

## 1. Objective and known portal result

Predict continuous annual turnover in INR. Score predictions using
`sqrt(mean((actual - prediction)**2))`. Lower RMSE is better.

The October 18 screenshots show **rank 18**, two submissions, and unchanged RMSE
**12,094,600.3216516**. Other entries moved ahead. The leader shown is
**submission**, RMSE **11,917,847.2693885**.
The required reduction is more than **176,753.05 (1.461%)** if that leading score
stays unchanged. This notebook's new CSV has **no observed portal score**.

This round screened 22 training settings on five-fold seed 42, tested six
shortlisted settings on seed 71, and selected a coarse blend using those two
development assignments. Settings were then frozen. The confirmation uses new
split assignments 18421/18422, previously used assignments 2026/303, and a separate
city-group stress test. All assignments reuse labels involved in earlier work:
these are robustness checks, not an independent external test or nested estimate.
No hidden labels or test-score-derived prediction corrections are used.

In [ ]:
import os,sys,json,hashlib,importlib.util,subprocess,warnings
from pathlib import Path
packages={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn','catboost':'catboost','scipy':'scipy','matplotlib':'matplotlib'}
missing=[v for k,v in packages.items() if importlib.util.find_spec(k) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn,catboost,scipy
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold,GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,SplineTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor,ExtraTreesRegressor
from sklearn.svm import SVR
warnings.filterwarnings('ignore',category=FutureWarning)
FILE_STEM='submission';OUTPUT_DIR=Path.cwd()
print('Runtime:',sys.version.split()[0])
print('Library versions:',{k:v.__version__ for k,v in {'numpy':np,'pandas':pd,'sklearn':sklearn,'catboost':catboost,'scipy':scipy}.items()})

## 2. Load inputs and check alignment

The row-level `Registration Number` is the restaurant ID. Student details belong
in the filename and notebook cover. Normalize the endorsement spelling difference
between train and test. Preserve the original test order and all validation targets.

In [ ]:
required=['Train_dataset.csv','Test_dataset.csv','Data_Dictionary.csv','Sample_Solution.csv']
DATA_DIR=Path.cwd()
if not all((DATA_DIR/n).exists() for n in required):
    local=Path('~/Downloads')
    if all((local/n).exists() for n in required):DATA_DIR=local
    else:
        from google.colab import files
        files.upload()
assert all((DATA_DIR/n).exists() for n in required)
train=pd.read_csv(DATA_DIR/required[0]).rename(columns={'Endorsed By':'Endoresed By'})
test=pd.read_csv(DATA_DIR/required[1]);dictionary=pd.read_csv(DATA_DIR/required[2]);sample=pd.read_csv(DATA_DIR/required[3])
y=train['Annual Turnover'].to_numpy(float)
assert len(test)==500 and np.isfinite(y).all() and (y>0).all()
assert train.drop(columns='Annual Turnover').columns.tolist()==test.columns.tolist()
assert train['Registration Number'].is_unique and test['Registration Number'].is_unique
assert set(train['Registration Number']).isdisjoint(test['Registration Number'])
assert sample.columns.tolist()==['Registration Number','Annual Turnover']
assert sample['Registration Number'].equals(test['Registration Number'])
input_hashes={n:hashlib.sha256((DATA_DIR/n).read_bytes()).hexdigest() for n in required}
print('PASS: input schemas, positive targets, unique restaurant IDs, disjoint train/test and sample row order.')
print('Train shape:',train.shape,'Test shape:',test.shape)
print('Input SHA256:',json.dumps(input_hashes,indent=2))

## 3. Modeling hypotheses

Extreme training turnovers have a large effect on squared error. This round compares
square-root/log/power target models, training-only mean corrections after inversion,
quantile objectives, training-only caps, feature selection inside each training fold,
and new feature combinations. Training changes never alter validation labels.
The correction, when selected, is the training mean of original target minus
inverse-transformed fitted prediction. It is estimated from training predictions;
it can be biased and must be judged by held-out original-scale RMSE.

No algorithm is assumed to improve results automatically. The complete selected
configuration and signed comparison results are printed below.

In [ ]:
print(pd.Series(y).describe(percentiles=[.5,.9,.95,.99]).to_string())
tail=y>100e6;squared=(y-y.mean())**2
print('Rows above INR100m:',tail.sum(),'Share of mean-baseline squared error:',100*squared[tail].sum()/squared.sum())
fig,ax=plt.subplots(figsize=(8,3.4));ax.hist(y/1e6,bins=45,color='#2563eb',edgecolor='white')
ax.set(xlabel='Annual turnover (INR millions)',ylabel='Restaurants, log scale',yscale='log',title='All original targets remain in validation')
fig.tight_layout();plt.show()

In [ ]:
TARGET = 'Annual Turnover'

ID = 'Registration Number'

EVENTS = ['Live Music Rating','Comedy Gigs Rating','Value Deals Rating','Live Sports Rating']

SOCIAL = ['Facebook Popularity Quotient','Instagram Popularity Quotient']

CORE = SOCIAL + ['Hygiene Rating','Restaurant Zomato Rating','Live Music Rating','Value for Money','Overall Restaurant Rating','Restaurant City Tier','Cuisine','Restaurant Theme','Restaurant Location','Endoresed By','City','Opening Year']

def features(raw, variant='base'):
    data=raw.drop(columns=[TARGET,ID],errors='ignore').copy()
    dates=pd.to_datetime(data.pop('Opening Day of Restaurant'),format='%d-%m-%Y',errors='coerce')
    data['Opening Year']=dates.dt.year.astype(float)
    data['Opening Month']=dates.dt.month.astype(float)
    for c in EVENTS:
        data['Offers '+c.replace(' Rating','')]=data[c].notna().astype(int)
    parts=data['Cuisine'].astype('string').fillna('Missing').str.split(',',n=1,expand=True)
    data['Cuisine 1']=parts[0].str.strip().str.lower()
    data['Cuisine 2']=parts.reindex(columns=[0,1])[1].astype('string').str.strip().str.lower()
    for c in ['Cuisine','City','Restaurant Location','Endoresed By','Restaurant Type','Restaurant Theme']:
        data[c]=data[c].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True).fillna('Missing')
    data['City Is Unknown']=(data['City']=='-1').astype(int)
    for c in data.select_dtypes(include=['object','string']).columns:
        data[c]=data[c].fillna('Missing').astype(str)
    if variant in ['engineered','engineered_drop']:
        data['City']=data['City'].replace({'bengaluru':'bangalore','gurugram':'gurgaon','bombay':'mumbai','calcutta':'kolkata','delhi':'new delhi'})
        data['Cuisine unordered']=['|'.join(sorted(v.split(','))) for v in data['Cuisine']]
        data['Opening Day']=dates.dt.day.astype(float)
        data['Opening Days Since 1970']=(dates-pd.Timestamp('1970-01-01')).dt.days.astype(float)
        data['Social mean']=data[SOCIAL].mean(axis=1)
        data['Social difference']=data[SOCIAL[0]]-data[SOCIAL[1]]
        data['Social missing count']=data[SOCIAL].isna().sum(axis=1)
        data['Rating missing count']=data.isna().sum(axis=1)
        data['Event count']=data[EVENTS].notna().sum(axis=1)
        data['Event mean']=data[EVENTS].mean(axis=1)
        data['Quality mean']=data[['Food Rating','Hygiene Rating','Overall Restaurant Rating']].mean(axis=1)
        data['Social x Zomato']=data['Social mean']*data['Restaurant Zomato Rating']
        data['Social x Hygiene']=data['Social mean']*data['Hygiene Rating']
        data['Quality x Value']=data['Quality mean']*data['Value for Money']
    if variant in ['drop_shift','engineered_drop']:
        data=data.drop(columns=['Service','Comfortablility'])
    if variant=='core': data=data[CORE]
    if variant=='ratings':
        keep=SOCIAL+['Resturant Tier','Restaurant Zomato Rating','Restaurant City Tier','Staff Responsivness','Value for Money','Hygiene Rating','Overall Restaurant Rating']+EVENTS+['Lively','Ambience','Privacy']
        data=data[keep]
    if variant=='no_city_date':
        data=data.drop(columns=['City','City Is Unknown','Opening Year','Opening Month'])
    return data

def rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))

def sklearn_model(X,spec,seed):
    cats=X.select_dtypes(include=['object','string']).columns.tolist()
    nums=[c for c in X if c not in cats]
    num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler())
    if spec['kind']=='spline':
        num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())
    pre=ColumnTransformer([('num',num,nums),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=10,sparse_output=False),cats)])
    kind=spec['kind']
    if kind in ['ridge','spline']: model=Ridge(alpha=spec.get('alpha',100))
    elif kind=='svr': model=SVR(C=spec.get('C',20),epsilon=.5,kernel='rbf')
    elif kind=='hist': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)
    elif kind=='extra': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)
    return make_pipeline(pre,model)

def fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):
    ys=np.asarray(y)[train_idx].copy()
    if 'cap' in spec: ys=np.minimum(ys,spec['cap'])
    if spec['kind']=='cat':
        params=dict(loss_function=spec.get('loss','RMSE'),eval_metric='RMSE',iterations=fixed_iterations or spec.get('iterations',1600),learning_rate=.04,depth=spec.get('depth',6),l2_leaf_reg=spec.get('l2',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)
        if 'boosting_type' in spec: params['boosting_type']=spec['boosting_type']
        if spec.get('monotone'): params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
        model=CatBoostRegressor(**params)
        cat=X.select_dtypes(include=['object','string']).columns.tolist()
        stopping={} if fixed_iterations or 'iterations' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)
        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)
        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_
    elif spec['kind'] in ['lgb','xgb']:
        pre=sklearn_model(X,dict(kind='ridge'),seed)[:-1]
        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])
        early=not (fixed_iterations or 'iterations' in spec)
        iterations=fixed_iterations or spec.get('iterations',1400)
        if spec['kind']=='lgb':
            from lightgbm import LGBMRegressor,early_stopping
            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get('leaves',15),min_child_samples=spec.get('leaf_samples',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get('objective','regression'))
            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric='rmse',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}
            model.fit(ztr,ys/1e6,**params)
            trees=model.best_iteration_ if early else iterations
        else:
            from xgboost import XGBRegressor
            params=dict(n_estimators=iterations,max_depth=spec.get('depth',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method='hist',objective=spec.get('objective','reg:squarederror'),eval_metric='rmse',base_score=float(ys.mean()/1e6))
            if early: params['early_stopping_rounds']=100
            if spec.get('objective')=='reg:pseudohubererror': params['huber_slope']=12
            model=XGBRegressor(**params)
            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))
            trees=model.best_iteration+1 if early else iterations
        p=model.predict(zva)*1e6
    else:
        model=sklearn_model(X,spec,seed)
        model.fit(X.iloc[train_idx],ys/1e6)
        p=model.predict(X.iloc[valid_idx])*1e6; trees=0
    if return_fitted:
        fitted=dict(model=model,preprocessor=pre if spec['kind'] in ['lgb','xgb'] else None,scale=1 if spec['kind']=='cat' else 1e6)
        return np.maximum(p,0),trees,fitted
    return np.maximum(p,0),trees

def predict_fitted(fitted,X):
    transformed=fitted['preprocessor'].transform(X) if fitted['preprocessor'] is not None else X
    return np.maximum(fitted['model'].predict(transformed)*fitted['scale'],0)

def cat_fit(X, y, spec, seed):
    params=dict(iterations=spec['iterations'], learning_rate=.04, depth=spec['depth'],
                l2_leaf_reg=spec.get('l2',5), random_seed=seed, thread_count=2,
                verbose=False, allow_writing_files=False,
                loss_function=f"Huber:delta={spec['delta']*1e6:g}" if 'delta' in spec else 'RMSE')
    if spec.get('monotone'):
        params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
    if spec.get('ordered'): params['boosting_type']='Ordered'
    if 'subsample' in spec: params.update(bootstrap_type='Bernoulli',subsample=spec['subsample'])
    if 'random_strength' in spec: params['random_strength']=spec['random_strength']
    fitted=CatBoostRegressor(**params)
    target=np.minimum(y,spec['cap']*1e6) if 'cap' in spec else y
    fitted.fit(X,target,cat_features=X.select_dtypes(include=['object','string']).columns.tolist())
    return fitted

def get_splits(train,scheme):
    if str(scheme)=='city':
        group=features(train,'engineered')['City']
        splits=list(GroupKFold(5).split(train,groups=group))
        for tr,va in splits:assert set(group.iloc[tr]).isdisjoint(group.iloc[va])
    else:splits=list(KFold(5,shuffle=True,random_state=int(scheme)).split(train))
    counts=np.zeros(len(train),int)
    for tr,va in splits:
        assert set(tr).isdisjoint(va);counts[va]+=1
    assert (counts==1).all()
    return splits

def compute_v3(train,y,scheme,contract):
    splits=get_splits(train,scheme);pred=np.zeros(len(y))
    for name,spec in contract['v2_configs'].items():
        if name not in contract['v2_weights']:continue
        X=features(train,spec['variant']);component=np.zeros(len(y))
        for f,(tr,va) in enumerate(splits):
            component[va],_=fit_predict(X,y,tr,va,spec,43+f,spec.get('iterations'))
        pred+=.5*contract['v2_weights'][name]*component
    X=features(train,'base');spec=contract['v3_parameters']
    for f,(tr,va) in enumerate(splits):
        parts=[]
        for seed in contract['v3_seeds']:
            model=cat_fit(X.iloc[tr],y[tr],spec,seed+f)
            parts.append(np.maximum(model.predict(X.iloc[va]),0))
        pred[va]+=.5*np.mean(parts,axis=0)
    return pred

def check_submission(frame,test):
    assert frame.shape==(500,2),'Expected 500 rows and two columns'
    assert frame.columns.tolist()==['Registration Number','Annual Turnover'],'Incorrect header'
    assert frame['Registration Number'].equals(test['Registration Number']),'Restaurant IDs/order changed'
    assert frame['Registration Number'].is_unique,'Duplicate restaurant IDs'
    assert np.isfinite(frame['Annual Turnover']).all(),'Non-finite prediction'
    assert (frame['Annual Turnover']>=0).all(),'Negative turnover'

def v5_features(raw,variant='base'):
    X=features(raw,'engineered' if variant=='rich' else 'base')
    if variant in ['clean','rich']:
        X['City']=X['City'].replace({'bengaluru':'bangalore','banglore':'bangalore','gurugram':'gurgaon','gurgoan':'gurgaon','bombay':'mumbai','calcutta':'kolkata','delhi':'new delhi','bhubneshwar':'bhubaneswar','hderabad':'hyderabad','punr':'pune'})
    if variant=='rich':
        fb=X['Facebook Popularity Quotient'];ig=X['Instagram Popularity Quotient']
        X['Social ratio']=fb/(ig+1)
        X['Social product']=fb*ig
        X['Social minimum']=np.minimum(fb,ig)
        X['Social absolute gap']=np.abs(fb-ig)
        X['Quality spread']=X[['Food Rating','Hygiene Rating','Overall Restaurant Rating']].std(axis=1)
        X['Zomato x Hygiene']=X['Restaurant Zomato Rating']*X['Hygiene Rating']
        X['City tier x Restaurant tier']=X['Restaurant City Tier']*X['Resturant Tier']
    return X

def inverse_target(z,power):
    if power==0:return np.expm1(np.minimum(z,20))
    return np.maximum(z,0)**(1/power)

def fit_v5(raw,y,spec,seed):
    X=v5_features(raw,spec.get('variant','base'));target=np.asarray(y,dtype=float)/1e6
    if 'cap' in spec:target=np.minimum(target,spec['cap'])
    power=spec.get('power',1.)
    transformed=np.log1p(target) if power==0 else target**power
    loss=spec.get('loss','RMSE')
    params=dict(iterations=spec.get('iterations',850),learning_rate=.04,depth=spec.get('depth',5),
                l2_leaf_reg=10,bootstrap_type='Bernoulli',subsample=.7,loss_function=loss,
                random_seed=seed,thread_count=2,verbose=False,allow_writing_files=False)
    if 'rsm' in spec:params['rsm']=spec['rsm']
    if 'topk' in spec:
        pilot=CatBoostRegressor(**{**params,'iterations':300})
        pilot.fit(X,transformed,cat_features=X.select_dtypes(include=['object','string']).columns.tolist())
        ranked=np.argsort(-pilot.feature_importances_);selected=[X.columns[i] for i in ranked[:spec['topk']]]
        X=X[selected]
    model=CatBoostRegressor(**params)
    model.fit(X,transformed,cat_features=X.select_dtypes(include=['object','string']).columns.tolist())
    correction=0.
    if spec.get('correct_mean'):
        # Training-only retransformation correction; all validation labels remain unseen.
        fitted_original=inverse_target(model.predict(X),power)
        correction=float(np.mean(target-fitted_original))
    return {'model':model,'columns':X.columns.tolist(),'power':power,'correction':correction,'spec':spec}

def predict_v5(fitted,raw):
    X=v5_features(raw,fitted['spec'].get('variant','base'))[fitted['columns']]
    return np.maximum(0,(inverse_target(fitted['model'].predict(X),fitted['power'])+fitted['correction'])*1e6)

def evaluate(train,y,scheme,contract,previous=None):
    if previous is None:previous=compute_v3(train,y,scheme,contract)
    prediction=contract['v3_weight']*previous
    parts={}
    for name,c in contract['components'].items():
        p=np.zeros(len(y))
        for f,(tr,va) in enumerate(get_splits(train,scheme)):
            one=[]
            for seed in contract['seeds']:
                fitted=fit_v5(train.iloc[tr],y[tr],c['spec'],seed+f)
                one.append(predict_v5(fitted,train.iloc[va]))
            p[va]=np.mean(one,axis=0)
        parts[name]=p;prediction+=c['weight']*p
    return {'previous':previous,'candidate':prediction,**parts}

def full_fit(train,test,y,contract):
    prior=np.zeros(len(test));fitted={'v2':{},'v3':[],'new':{}}
    tr=np.arange(len(train));va=np.arange(len(train),len(train)+len(test));yy=np.r_[y,np.full(len(test),np.nan)]
    for name,w in contract['v2_weights'].items():
        spec=contract['v2_configs'][name]
        X=pd.concat([features(train,spec['variant']),features(test,spec['variant'])],ignore_index=True)
        p,_,model=fit_predict(X,yy,tr,va,spec,42,spec.get('iterations'),return_fitted=True)
        fitted['v2'][name]=model;prior+=.5*w*p
    for seed in contract['v3_seeds']:
        model=cat_fit(features(train,'base'),y,contract['v3_parameters'],seed);fitted['v3'].append(model)
        prior+=.5*np.maximum(model.predict(features(test,'base')),0)/len(contract['v3_seeds'])
    candidate=contract['v3_weight']*prior
    for name,c in contract['components'].items():
        fitted['new'][name]=[]
        for seed in contract['seeds']:
            model=fit_v5(train,y,c['spec'],seed);fitted['new'][name].append(model)
            candidate+=c['weight']*predict_v5(model,test)/len(contract['seeds'])
    return prior,candidate,fitted

def predict_final(raw,fitted,contract):
    prior=sum(.5*w*predict_fitted(fitted['v2'][name],features(raw,contract['v2_configs'][name]['variant'])) for name,w in contract['v2_weights'].items())
    prior+=.5*np.mean([np.maximum(m.predict(features(raw,'base')),0) for m in fitted['v3']],axis=0)
    p=contract['v3_weight']*prior
    for name,c in contract['components'].items():p+=c['weight']*np.mean([predict_v5(m,raw) for m in fitted['new'][name]],axis=0)
    return p

In [ ]:
CONTRACT=json.loads('{"v2_configs": {"engineered": {"kind": "cat", "variant": "engineered", "depth": 6, "iterations": 260}, "engineered_drop": {"kind": "cat", "variant": "engineered_drop", "depth": 6, "iterations": 216}, "ordered_depth5": {"kind": "cat", "variant": "base", "depth": 5, "boosting_type": "Ordered", "iterations": 219}, "huber20_engineered": {"kind": "cat", "variant": "engineered_drop", "depth": 5, "loss": "Huber:delta=20000000", "iterations": 699}, "huber40": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 664}, "clip120": {"kind": "cat", "variant": "base", "depth": 5, "cap": 120000000.0, "iterations": 591}, "spline_ridge": {"kind": "spline", "variant": "base", "alpha": 100}, "svr_core": {"kind": "svr", "variant": "core", "C": 20}, "huber40_engineered": {"kind": "cat", "variant": "engineered", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 619}, "huber40_monotone": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "monotone": true, "iterations": 447}}, "v2_weights": {"engineered": 0.06448907249408362, "engineered_drop": 0.09631834163153197, "ordered_depth5": 0.08507039177778712, "huber20_engineered": 0.06274452861282048, "huber40": 0.08855362607985372, "clip120": 0.07794631558814363, "spline_ridge": 0.15648185431084338, "svr_core": 0.058428294374292714, "huber40_engineered": 0.09327747966602483, "huber40_monotone": 0.2166900954646185}, "v3_parameters": {"variant": "base", "depth": 5, "iterations": 850, "delta": 40, "subsample": 0.7, "l2": 10}, "v3_seeds": [1042, 2042, 3042], "version": "v5", "v3_weight": 0.25, "seeds": [10242, 20242], "components": {"sqrt_d6_correct1": {"spec": {"power": 0.5, "depth": 6, "correct_mean": true}, "weight": 0.25}, "subspace_h40": {"spec": {"loss": "Huber:delta=40", "rsm": 0.5, "iterations": 1200}, "weight": 0.25}, "quantile0.6": {"spec": {"loss": "Quantile:alpha=0.6", "depth": 5}, "weight": 0.25}}, "development": {"settings": 22, "shortlisted_settings": 6, "splits": [42, 71], "selection": {"name": "top3", "new_weight": 0.75, "rmse42": 19817236.327581998, "rmse71": 19840250.361659143, "mean_rmse": 19828743.34462057, "gain42": 21791.213259298354, "gain71": 29833.246005315334}, "screening_is_optimistic": true}, "confirmation": {"new_splits": [18421, 18422], "previous_splits": [2026, 303], "city_stress": true, "all_splits_reuse_previously_seen_labels": true}}')

assert np.isclose(CONTRACT['v3_weight']+sum(c['weight'] for c in CONTRACT['components'].values()),1)
assert np.isclose(sum(CONTRACT['v2_weights'].values()),1)
assert np.isclose(rmse([0,3],[0,0]),np.sqrt(4.5))
changed=train.copy();changed['Annual Turnover']=0;changed['Registration Number']=-1
for variant in ['base','clean','rich']:
    X=v5_features(train,variant)
    assert X.columns.equals(v5_features(test,variant).columns)
    pd.testing.assert_frame_equal(X,v5_features(changed,variant))
    assert 'Annual Turnover' not in X and 'Registration Number' not in X
print('PASS: RMSE toy example, weight sums, target/ID exclusion and feature invariance.')
print('Frozen blend:',CONTRACT['v3_weight'],json.dumps(CONTRACT['components'],indent=2))
print('Development selection:',json.dumps(CONTRACT['development'],indent=2))

## 4. Same-fold comparisons and uncertainty

The local executed copy may use explicitly requested actual fold predictions with
verified input/configuration/file hashes. It recomputes every metric. Colab normally
has no cache and trains all folds. Each split scores every original target exactly
once. Known city aliases are grouped together for the separate city stress check;
other spelling variants may remain, so this is an approximate geographic check.

In [ ]:
cache_value=os.environ.get('MLA_VALIDATION_CACHE');cache_root=Path(cache_value) if cache_value else None
manifest=None
if cache_root is not None:
    manifest=json.loads((cache_root/'cache_manifest.json').read_text())
    assert manifest['inputs']==input_hashes
    assert manifest['contract']==hashlib.sha256(json.dumps(CONTRACT,sort_keys=True).encode()).hexdigest()
validation_predictions={};rows=[]
for scheme in [18421,18422,2026,303,'city']:
    get_splits(train,scheme)
    if cache_root is None:
        print('Training validation assignment:',scheme,flush=True)
        predictions=evaluate(train,y,scheme,CONTRACT)
    else:
        name=f'comparison_{scheme}.npz'
        assert hashlib.sha256((cache_root/name).read_bytes()).hexdigest()==manifest['files'][name]
        predictions=dict(np.load(cache_root/name));print('Verified actual fold cache:',name)
    for key in ['previous','candidate']:
        assert predictions[key].shape==y.shape and np.isfinite(predictions[key]).all() and (predictions[key]>=0).all()
    validation_predictions[str(scheme)]=predictions
    rows.append({'Validation':str(scheme),'Previous RMSE':rmse(y,predictions['previous']),'Candidate RMSE':rmse(y,predictions['candidate'])})
validation_table=pd.DataFrame(rows);validation_table['Reduction']=validation_table['Previous RMSE']-validation_table['Candidate RMSE']
print(validation_table.to_string(index=False,float_format=lambda x:f'{x:,.2f}'))
random_table=validation_table[validation_table.Validation!='city']
means=random_table.drop(columns='Validation').mean()
fresh_means=validation_table[validation_table.Validation.isin(['18421','18422'])].drop(columns='Validation').mean()
print('Mean four shuffled assignments:',means.to_dict())
print('Mean two new assignments:',fresh_means.to_dict())
supported=bool(means['Reduction']>0 and fresh_means['Reduction']>0)
print('Recommendation:', 'Reasonable candidate to try on the portal; ranking remains unconfirmed.' if supported else 'Keep confirmed v3 as preferred; this candidate is experimental.')

In [ ]:
schemes=[18421,18422,2026,303]
previous_sq=np.mean([(y-validation_predictions[str(s)]['previous'])**2 for s in schemes],axis=0)
candidate_sq=np.mean([(y-validation_predictions[str(s)]['candidate'])**2 for s in schemes],axis=0)
rng=np.random.default_rng(812);gains=[]
for _ in range(2000):
    ix=rng.integers(0,len(y),len(y));gains.append(np.sqrt(previous_sq[ix].mean())-np.sqrt(candidate_sq[ix].mean()))
gain_ci=np.quantile(gains,[.025,.975]);print('Conditional row-bootstrap95% interval for reduction:',gain_ci.tolist())
print('Interval includes zero:',bool(gain_ci[0]<=0<=gain_ci[1]))
print('This conditional interval excludes full model-selection and distribution uncertainty. Repeated splits reuse restaurants.')
fig,axes=plt.subplots(1,2,figsize=(12,4.5));x=np.arange(5);labels=['18421*','18422*','2026','303','City']
for column,color in [('Previous RMSE','#2563eb'),('Candidate RMSE','#0f766e')]:
    axes[0].plot(x[:4],validation_table[column].iloc[:4]/1e6,'o-',color=color,label=column)
    axes[0].scatter(x[-1],validation_table[column].iloc[-1]/1e6,color=color,marker='D')
axes[0].set(ylabel='RMSE (INR millions)',title='Same rows, original turnover values');axes[0].legend()
reduction=validation_table.Reduction.to_numpy()/1000
axes[1].bar(x,reduction,color=['#0f766e' if r>0 else '#b45309' for r in reduction]);axes[1].axhline(0,color='#334155')
axes[1].set(ylabel='RMSE reduction (INR thousands)',title='Positive = better; negative = worse')
for ax in axes:
    ax.set_xticks(x,labels);ax.set_xlabel('*New split; city excluded from random averages');ax.axvline(3.5,linestyle='--',color='#64748b')
fig.tight_layout();plt.show()

## 5. Train all selected models and test the export

Full fitting uses all supplied training rows. The optional independent full-fit
reference is only checked after refitting; predictions are not substituted from it.
Format and behavior checks do not measure accuracy on hidden test targets.

In [ ]:
previous_test,test_prediction,fitted=full_fit(train,test,y,CONTRACT)
assert np.allclose(predict_final(test,fitted,CONTRACT),test_prediction,rtol=1e-12,atol=.01)
if cache_root is not None:
    name='full_prediction_parts.npz'
    assert hashlib.sha256((cache_root/name).read_bytes()).hexdigest()==manifest['files'][name]
    expected=np.load(cache_root/name)
    assert np.allclose(previous_test,expected['previous'],rtol=1e-12,atol=.01)
    assert np.allclose(test_prediction,expected['candidate'],rtol=1e-12,atol=.01)
perm=np.random.default_rng(91).permutation(len(test))
assert np.allclose(predict_final(test.iloc[perm],fitted,CONTRACT),test_prediction[perm],rtol=1e-10,atol=.01)
edge=test.iloc[:2].copy();edge['City']='unseen_city';edge['Cuisine']=pd.NA;edge['Facebook Popularity Quotient']=np.nan
assert np.isfinite(predict_final(edge,fitted,CONTRACT)).all()
submission=pd.DataFrame({'Registration Number':test['Registration Number'],'Annual Turnover':test_prediction});check_submission(submission,test)
bad=[submission.iloc[:-1],submission.iloc[::-1].reset_index(drop=True),submission.rename(columns={'Annual Turnover':'Turnover'}),submission.assign(index=0),submission.assign(**{'Registration Number':'submission'})]
for value in [np.nan,np.inf,-1]:
    one=submission.copy();one.loc[0,'Annual Turnover']=value;bad.append(one)
for one in bad:
    rejected=False
    try:check_submission(one,test)
    except (AssertionError,TypeError,ValueError):rejected=True
    assert rejected
output_path=OUTPUT_DIR/(FILE_STEM+'.csv');submission.to_csv(output_path,index=False)
roundtrip=pd.read_csv(output_path);check_submission(roundtrip,test)
assert np.allclose(roundtrip['Annual Turnover'],test_prediction,rtol=1e-12,atol=.001)
assert len(output_path.read_text().splitlines())==501
print('PASS: full-fit reproduction, row permutation, missing/unseen values, eight invalid examples, CSV round-trip.')
print('Saved:',output_path.name,'Rows:',len(submission),'Columns:',submission.columns.tolist())
print('SHA256:',hashlib.sha256(output_path.read_bytes()).hexdigest())
print(submission.head().to_string(index=False))
print('Mean absolute change from confirmed v3:',float(np.abs(test_prediction-previous_test).mean()))

## 6. Inferences and submission

Use the signed validation results and uncertainty above to judge this candidate.
Local RMSE differs from the portal RMSE because the evaluated restaurants differ.
Do not subtract a local improvement from the prior portal score to predict ranking.
The new CSV's portal RMSE is unknown until it is evaluated there. First place would
require a score below **11,917,847.2693885**, if the shown leader is unchanged.

If choosing to try this candidate, upload the CSV to the hackathon module. Record
the returned score with its exact filename/version/hash. Preserve the previously
successful CSV. Submit the executed notebook matching the final chosen model to
**LLJ_Oct_2026** when ready; the provided instructions allow one notebook submission.

In [ ]:
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:print('CSV is ready in the notebook folder.')

## Measured result of this executed local run

**Candidate recommended for a new portal trial.** Mean RMSE reduction across four shuffled assignments: **31,890.83**.
Across the two new split assignments: **34,553.02**.
Conditional bootstrap interval: **[3,832.21, 56,418.51]**.

One complete five-fold assignment was also retrained through this notebook's
embedded code without prediction caches. Its predictions matched the experiment
reference within INR0.01. All selected full-data models were freshly fitted in
this notebook, and the resulting CSV matched the independent full fit.

These are local validation results. The candidate has no observed portal score,
and first place is not established. The confirmed prior portal score remains
12,094,600.3216516; the October18 leading score is 11,917,847.2693885.
